# 🍷 Wine Quality Prediction - Progetto Capstone

## Contesto del Progetto

### Il Cliente: Cantina di Pregio Portoghese

La nostra cantina cliente, produttrice di **Vinho Verde** (vini verdi portoghesi), si trova di fronte a una sfida cruciale nel processo di produzione: **quale lotti di vino destinare all'affinamento in barrique** per ottenere vini di alta qualità destinati al mercato premium.

### La Richiesta dell'Enologo

L'enologo capo della cantina ci ha contattato con una richiesta specifica:

> *"Ogni anno produciamo centinaia di lotti di vino. Attualmente, la decisione su quali lotti far invecchiare nelle costose botti di rovere si basa principalmente sull'esperienza e sull'assaggio. Questo processo è:*
> - *Soggettivo e dipendente dalla disponibilità dell'enologo*
> - *Costoso in termini di tempo (richiede settimane per le valutazioni)*
> - *Rischioso: investiamo in barrique costose senza certezza del risultato finale*
>
> *Abbiamo bisogno di un sistema che, analizzando i parametri chimico-fisici del vino in fase iniziale, predica con affidabilità se un lotto merita l'investimento per l'affinamento premium."*

### Raccolta Dati

La cantina ha messo a disposizione il proprio **database storico** contenente:
- **Analisi chimico-fisiche** di oltre 6,000 campioni di vino
- **Valutazioni sensoriali** da parte di panel di esperti sommelier (score 0-10)
- Parametri misurati in laboratorio: acidità, pH, solfiti, alcol, residui zuccherini, ecc.

I dati provengono da:
1. **Laboratorio interno** della cantina (misure oggettive)
2. **Panel di degustazione certificato** (valutazione qualità)
3. **Archivio storico** delle decisioni di affinamento e risultati commerciali

### Dataset: UCI Wine Quality

Per questo progetto utilizziamo il dataset pubblico **UCI Wine Quality** (ID: 186), che rappresenta esattamente lo scenario descritto:
- **6,497 campioni** di vini Vinho Verde (rossi e bianchi)
- **11 variabili chimico-fisiche** misurate in laboratorio
- **Valutazione qualitativa** (0-10) da esperti enologi

### Obiettivo del Progetto

Sviluppare un **sistema di Machine Learning** che:
1. Predica la **probabilità che un vino sia di alta qualità** (score ≥ 7)
2. Supporta l'enologo nelle **decisioni di affinamento** 

### Deliverable

1. **Modello ML** addestrato e validato (pipeline scikit-learn)
2. **Webapp Streamlit** per l'utilizzo in cantina
3. **Sistema MLflow** per tracking e versionamento
4. **Explainability** (SHAP, Feature Importance) per trasparenza decisionale

---

## Esplorazione Dataset e Sviluppo Modello

In [ ]:
from ucimlrepo import fetch_ucirepo 
import os
import mlflow

# Opt-in necessario nelle versioni recenti per usare il backend file legacy.
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
mlflow.set_tracking_uri("file:mlruns")  


# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
X = wine_quality.data.features 
y = wine_quality.data.targets 
  
# metadata 
print(wine_quality.metadata) 
  
# variable information 
print(wine_quality.variables) 


In [ ]:
# Binarizza la variabile y: 1 se quality >= 7, altrimenti 0
y_bin = (y['quality'] >= 7).astype(int)
# rename in high_quality
y_bin.rename('high_quality', inplace=True)
print(y_bin.value_counts())

---

## 📋 Guida al Progetto

### Requisiti Minimi 

Per completare il progetto, è **necessario** eseguire i seguenti passi:

1. ✅ **Allenare un modello di Machine Learning**
   - Utilizzare i dati del wine quality dataset
   - Creare una pipeline scikit-learn completa (preprocessing + model)
   - Valutare le performance del modello

2. ✅ **Loggare il modello su MLflow**
   - Configurare MLflow tracking
   - Loggare parametri, metriche e artefatti del modello
   - Salvare la pipeline completa con `mlflow.sklearn.log_model(..., serialization_format="cloudpickle")` (come in T4)

   💡 La sidebar della webapp mostra le metriche **solo** se sono loggate con questi nomi esatti: `accuracy`, `precision`, `recall`, `f1_score`, `roc_auc`

3. ✅ **Registrare il modello nel Model Registry**
   - Registrare il modello con nome `wine_clf`
   - Assegnare l'alias `@production` alla versione migliore
   - Verificare che il modello sia accessibile via registry

4. ✅ **Testare la Webapp**
   - Eseguire `uv run streamlit run app.py` dalla cartella `capstone_project`
   - Verificare che carichi correttamente il modello
   - Testare le predizioni con diversi parametri

---

### Passi Avanzati (Opzionali ma Consigliati)

Una volta completati i requisiti minimi, potete approfondire con:

#### **1. Valutazione Avanzata delle Performance**

**Metriche sul Test Set:**
- Calcolare **ROC-AUC** per valutare la capacità discriminativa
- Analizzare **Precision-Recall Curve** per trovare il trade-off ottimale
- Confrontare diverse soglie di classificazione

**Domanda Critica da Rispondere:**
> Nel contesto della cantina, **quale errore è più costoso?**
> - **Falso Positivo**: Affinare in barrique un vino che poi risulta mediocre → spreco investimento
> - **Falso Negativo**: Non affinare un vino eccellente → perdita opportunità di vendita premium
> 
> In base alla risposta, ottimizzare per **Precision** (minimizzare FP) o **Recall** (minimizzare FN)

#### **2. Model Comparison e Selection**

- **Testare diversi algoritmi**:
  - Random Forest (baseline)
  - Logistic Regression (interpretabilità)
  - SVM con kernel RBF
  
- **Loggare tutti i modelli su MLflow** per confronto sistematico
- **Selezionare il migliore** in base a metrica rilevante + interpretabilità

#### **3. Explainability e Feature Importance**

**Permutation Feature Importance (PFI):**
- Identificare quali feature contribuiscono maggiormente alle predizioni
- Loggare plot PFI come artefatto MLflow

**SHAP Values:**
- Calcolare SHAP values per interpretabilità locale
- Generare summary plot e dependence plot
- Loggare visualizzazioni su MLflow

**Domande da Esplorare:**
- Quali parametri chimico-fisici sono più importanti per la qualità?
- Ci sono interazioni tra feature? (es. alcol + acidità)
- Il modello è allineato con la conoscenza enologica?

#### **4. Iterazione e Miglioramento**

1. **Esplorare le diverse run MLflow**:
   - Usare MLflow UI: `MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri file:mlruns --port 5000` (dalla cartella `capstone_project`)
   - Confrontare run parallele
   - Identificare pattern nei parametri vincenti

2. **Aggiornare la webapp**:
   - Registrare il modello migliorato con alias `@production`
   - Riavviare la webapp
   - Verificare che le predizioni siano migliorate

3. **Documentare il processo**:
   - Motivare la scelta del modello finale
   - Spiegare il trade-off precision/recall scelto
   - Documentare le feature più importanti

---

**Iniziate con i requisiti minimi, poi espandete progressivamente con i passi avanzati!**

In [ ]:
# iniziate qui ...

### Nota Importante

Il codice riportato sotto deve essere eseguito **alla fine** del workflow. Questo passaggio è necessario per salvare il modello migliore in una cartella locale, che potrà essere utilizzata per il **deploy** su **Streamlit Cloud**. 

Assicurati che il modello sia stato registrato e che la versione in produzione sia stata correttamente selezionata prima di eseguire il codice.

In [ ]:
import shutil
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# Inizializza client MLflow
client = MlflowClient()
# Nome del modello registrato
model_name = "wine_clf"

model_uri = f"models:/{model_name}@production"

# Carica il modello
model = mlflow.sklearn.load_model(model_uri)

# Salva il modello in una cartella locale (model/model.pkl, letto dalla webapp come fallback)
shutil.rmtree("model", ignore_errors=True)  # sovrascrive un eventuale export precedente
mlflow.sklearn.save_model(model, "model", serialization_format="cloudpickle")
